In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import timm


from paddy_10_data_loader import load_train_val_data
from shufflenet_v2 import ShuffleNetV2
from helper_utils import training_loop

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## Comparison of Lightweight models

### 1. Parameter Count

In [4]:
def count_params(model):

    # 1. Trainable Parameters (requires_grad = True)
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

    # 2. Non-Trainable Parameters 
    # This includes frozen parameters AND buffers (like BN moving stats) to match TF
    frozen_params = sum(p.numel() for p in model.parameters() if not p.requires_grad)
    buffers = sum(b.numel() for b in model.buffers())
    non_trainable_total = frozen_params + buffers

    # 3. Total Parameters
    total_params = trainable_params + non_trainable_total

    print(f"{'Total Parameters:':<25} {total_params:,}")
    print(f"{'Trainable Parameters:':<25} {trainable_params:,}")
    print(f"{'Non-trainable Parameters:':<25} {non_trainable_total:,}")

In [5]:
shufflenetV2_standalone = ShuffleNetV2(n_class=10, model_size='1.0x')
fasternet_standalone = timm.create_model('fasternet_t0.in1k', pretrained=False, num_classes=10)
mobilenetV4_standalone = timm.create_model('mobilenetv4_conv_small.e2400_r224_in1k', pretrained=False, num_classes=10)

count_params(shufflenetV2_standalone)
count_params(fasternet_standalone)
count_params(mobilenetV4_standalone)

model size is  1.0x
Total Parameters:         1,281,236
Trainable Parameters:     1,265,000
Non-trainable Parameters: 16,236
Total Parameters:         2,647,007
Trainable Parameters:     2,637,310
Non-trainable Parameters: 9,697
Total Parameters:         2,530,968
Trainable Parameters:     2,505,834
Non-trainable Parameters: 25,134


### 2. Performance comparison between lightweight models

In [6]:
EPOCHS = 15
BATCH_SIZE = 32
LR = 0.001
NUM_CLASSES = 10

In [7]:
shufflenetV2_acc = []
fasternet_acc = []
mobilenetV4_acc = []

In [8]:
for run in range(5):
    print(f"Run {run + 1}/5")
    # Load data
    train_loader, val_loader = load_train_val_data(batch_size=BATCH_SIZE)
    
    # Train ShuffleNetV2 standalone
    shufflenetV2_standalone = ShuffleNetV2(n_class=NUM_CLASSES, model_size='1.0x')
    shufflenetV2_standalone_loss = nn.CrossEntropyLoss() 
    shufflenetV2_standalone_optimizer = optim.Adam(shufflenetV2_standalone.parameters(), lr=LR)
    _, shufflenet_standalone_history, = training_loop(
    model=shufflenetV2_standalone,
    train_loader=train_loader,
    val_loader=val_loader,
    loss_function=shufflenetV2_standalone_loss,
    optimizer=shufflenetV2_standalone_optimizer,
    num_epochs=EPOCHS,
    device=device,
    scheduler=None,
    save_path=None,
    )
    shufflenetV2_acc.append(shufflenet_standalone_history['val_accuracy'][-1])
    
    
    # Train Fasternet standalone
    fasternet_standalone = timm.create_model('fasternet_t0.in1k', pretrained=False, num_classes=NUM_CLASSES)
    fasternet_standalone_loss = nn.CrossEntropyLoss()
    fasternet_standalone_optimizer = optim.Adam(fasternet_standalone.parameters(), lr=LR)
    _, fasternet_standalone_history = training_loop(
    model=fasternet_standalone,
    train_loader=train_loader,
    val_loader=val_loader,
    loss_function=fasternet_standalone_loss,
    optimizer=fasternet_standalone_optimizer,
    num_epochs=EPOCHS,
    device=device,
    scheduler=None,
    save_path=None,
    )
    fasternet_acc.append(fasternet_standalone_history['val_accuracy'][-1])

    # train mobilenetV4 standalone
    mobilenetV4_standalone = timm.create_model('mobilenetv4_conv_small.e2400_r224_in1k', pretrained=False, num_classes=NUM_CLASSES)
    mobilenetV4_standalone_loss = nn.CrossEntropyLoss()
    mobilenetV4_standalone_optimizer = optim.Adam(mobilenetV4_standalone.parameters(), lr=LR)
    _, mobilenetV4_standalone_history = training_loop(
    model=mobilenetV4_standalone,
    train_loader=train_loader,
    val_loader=val_loader,
    loss_function=mobilenetV4_standalone_loss,
    optimizer=mobilenetV4_standalone_optimizer,
    num_epochs=EPOCHS,
    device=device,
    scheduler=None,
    save_path=None,
    )
    mobilenetV4_acc.append(mobilenetV4_standalone_history['val_accuracy'][-1])

Run 1/5
model size is  1.0x


Overall Progress:   0%|          | 0/4905 [00:00<?, ?it/s]

Epoch 1/15 - Train Acc: 33.8820%, Train Loss: 1.8797, Val Loss: 1.7639, Val Acc: 38.3877%
Epoch 2/15 - Train Acc: 51.8203%, Train Loss: 1.4107, Val Loss: 1.2285, Val Acc: 59.5969%
Epoch 3/15 - Train Acc: 64.4359%, Train Loss: 1.0757, Val Loss: 1.1150, Val Acc: 63.7716%
Epoch 4/15 - Train Acc: 73.8676%, Train Loss: 0.7938, Val Loss: 0.8467, Val Acc: 73.8964%
Epoch 5/15 - Train Acc: 80.5839%, Train Loss: 0.6050, Val Loss: 1.0837, Val Acc: 67.0345%
Epoch 6/15 - Train Acc: 84.8732%, Train Loss: 0.4496, Val Loss: 0.7645, Val Acc: 75.5278%
Epoch 7/15 - Train Acc: 89.1505%, Train Loss: 0.3352, Val Loss: 0.6309, Val Acc: 80.3263%
Epoch 8/15 - Train Acc: 91.4454%, Train Loss: 0.2603, Val Loss: 0.6987, Val Acc: 80.4703%
Epoch 9/15 - Train Acc: 91.8179%, Train Loss: 0.2613, Val Loss: 0.5664, Val Acc: 83.7332%
Epoch 10/15 - Train Acc: 93.2596%, Train Loss: 0.1958, Val Loss: 0.8669, Val Acc: 77.7351%
Epoch 11/15 - Train Acc: 93.9565%, Train Loss: 0.1818, Val Loss: 0.5467, Val Acc: 84.7889%
Epoch 12

Overall Progress:   0%|          | 0/4905 [00:00<?, ?it/s]

Epoch 1/15 - Train Acc: 29.2082%, Train Loss: 1.9601, Val Loss: 1.8771, Val Acc: 31.2380%
Epoch 2/15 - Train Acc: 38.0272%, Train Loss: 1.7332, Val Loss: 1.9010, Val Acc: 33.1094%
Epoch 3/15 - Train Acc: 42.9893%, Train Loss: 1.6016, Val Loss: 1.9496, Val Acc: 39.7793%
Epoch 4/15 - Train Acc: 46.1132%, Train Loss: 1.5317, Val Loss: 2.0159, Val Acc: 35.4127%
Epoch 5/15 - Train Acc: 50.8230%, Train Loss: 1.4219, Val Loss: 1.7577, Val Acc: 44.1459%
Epoch 6/15 - Train Acc: 54.5116%, Train Loss: 1.3271, Val Loss: 1.4538, Val Acc: 50.1440%
Epoch 7/15 - Train Acc: 56.4941%, Train Loss: 1.2725, Val Loss: 1.7081, Val Acc: 44.2418%
Epoch 8/15 - Train Acc: 59.8943%, Train Loss: 1.1602, Val Loss: 1.1935, Val Acc: 57.0058%
Epoch 9/15 - Train Acc: 62.0089%, Train Loss: 1.0835, Val Loss: 1.1335, Val Acc: 61.8042%
Epoch 10/15 - Train Acc: 65.5052%, Train Loss: 0.9990, Val Loss: 0.9815, Val Acc: 66.8426%
Epoch 11/15 - Train Acc: 67.4997%, Train Loss: 0.9582, Val Loss: 0.9121, Val Acc: 69.4818%
Epoch 12

...

Epoch 1/15 - Train Acc: 33.5336%, Train Loss: 1.8848, Val Loss: 2.2850, Val Acc: 33.1574%
Epoch 2/15 - Train Acc: 51.7602%, Train Loss: 1.3963, Val Loss: 1.3581, Val Acc: 54.0307%
Epoch 3/15 - Train Acc: 63.9673%, Train Loss: 1.0675, Val Loss: 1.0248, Val Acc: 66.2668%
Epoch 4/15 - Train Acc: 73.2548%, Train Loss: 0.8044, Val Loss: 0.8984, Val Acc: 69.7217%
Epoch 5/15 - Train Acc: 80.8723%, Train Loss: 0.5909, Val Loss: 0.8173, Val Acc: 74.5202%
Epoch 6/15 - Train Acc: 85.2577%, Train Loss: 0.4583, Val Loss: 0.7352, Val Acc: 76.4395%
Epoch 7/15 - Train Acc: 88.3936%, Train Loss: 0.3470, Val Loss: 0.7059, Val Acc: 80.2303%
Epoch 8/15 - Train Acc: 90.9288%, Train Loss: 0.2802, Val Loss: 0.6363, Val Acc: 81.9578%
Epoch 9/15 - Train Acc: 92.4426%, Train Loss: 0.2348, Val Loss: 0.6316, Val Acc: 81.7178%
Epoch 10/15 - Train Acc: 94.1968%, Train Loss: 0.1758, Val Loss: 0.5692, Val Acc: 83.4933%
Epoch 11/15 - Train Acc: 93.5120%, Train Loss: 0.1989, Val Loss: 0.5630, Val Acc: 84.7409%
Epoch 12

Overall Progress:   0%|          | 0/4905 [00:00<?, ?it/s]

Epoch 1/15 - Train Acc: 28.7637%, Train Loss: 1.9892, Val Loss: 2.1563, Val Acc: 25.5278%
Epoch 2/15 - Train Acc: 38.1233%, Train Loss: 1.7397, Val Loss: 1.9551, Val Acc: 36.6123%
Epoch 3/15 - Train Acc: 43.2176%, Train Loss: 1.6226, Val Loss: 2.0084, Val Acc: 36.6123%
Epoch 4/15 - Train Acc: 46.4136%, Train Loss: 1.5349, Val Loss: 1.6693, Val Acc: 43.8580%
Epoch 5/15 - Train Acc: 50.8470%, Train Loss: 1.4217, Val Loss: 1.4459, Val Acc: 50.5758%
Epoch 6/15 - Train Acc: 55.1364%, Train Loss: 1.3194, Val Loss: 1.3219, Val Acc: 54.7505%
Epoch 7/15 - Train Acc: 57.1308%, Train Loss: 1.2472, Val Loss: 1.2407, Val Acc: 56.9098%
Epoch 8/15 - Train Acc: 59.1734%, Train Loss: 1.1942, Val Loss: 1.4309, Val Acc: 50.5758%
Epoch 9/15 - Train Acc: 62.0930%, Train Loss: 1.1109, Val Loss: 1.2835, Val Acc: 57.8215%
Epoch 10/15 - Train Acc: 64.7603%, Train Loss: 1.0425, Val Loss: 1.1473, Val Acc: 63.6756%
Epoch 11/15 - Train Acc: 66.4304%, Train Loss: 0.9866, Val Loss: 0.9997, Val Acc: 64.2035%
Epoch 12

Overall Progress:   0%|          | 0/4905 [00:00<?, ?it/s]

Epoch 1/15 - Train Acc: 26.8413%, Train Loss: 2.4957, Val Loss: 2.1627, Val Acc: 25.4319%
Epoch 2/15 - Train Acc: 40.9227%, Train Loss: 1.7677, Val Loss: 1.8225, Val Acc: 43.6180%
Epoch 3/15 - Train Acc: 50.9071%, Train Loss: 1.4843, Val Loss: 1.8011, Val Acc: 47.5048%
Epoch 4/15 - Train Acc: 61.5884%, Train Loss: 1.1813, Val Loss: 1.1976, Val Acc: 61.9482%
Epoch 5/15 - Train Acc: 69.6143%, Train Loss: 0.9376, Val Loss: 1.2494, Val Acc: 61.0845%
Epoch 6/15 - Train Acc: 76.2105%, Train Loss: 0.7470, Val Loss: 1.6663, Val Acc: 59.7409%
Epoch 7/15 - Train Acc: 81.5932%, Train Loss: 0.5825, Val Loss: 1.0129, Val Acc: 70.9213%
Epoch 8/15 - Train Acc: 85.7623%, Train Loss: 0.4561, Val Loss: 1.5349, Val Acc: 65.6910%
Epoch 9/15 - Train Acc: 88.1293%, Train Loss: 0.3822, Val Loss: 2.3129, Val Acc: 54.1267%
Epoch 10/15 - Train Acc: 88.3335%, Train Loss: 0.3687, Val Loss: 0.9401, Val Acc: 75.2399%
Epoch 11/15 - Train Acc: 90.8326%, Train Loss: 0.2879, Val Loss: 3.3632, Val Acc: 49.0403%
Epoch 12

In [9]:
print(shufflenetV2_acc)
print(fasternet_acc)
print(mobilenetV4_acc)

[0.8809980750083923, 0.8651631474494934, 0.8550863862037659, 0.8522073030471802, 0.8445297479629517]
[0.665067195892334, 0.7236084342002869, 0.7269673943519592, 0.7442418336868286, 0.7380038499832153]
[0.7394433617591858, 0.7691938877105713, 0.7317658066749573, 0.7068138122558594, 0.8181381821632385]


In [10]:
np.mean(shufflenetV2_acc),  np.mean(fasternet_acc), np.mean(mobilenetV4_acc)

(np.float64(0.8595969319343567),
 np.float64(0.7195777416229248),
 np.float64(0.7530710101127625))